In [12]:
%pip -q install beautifulsoup4 trafilatura

import re, json, time, hashlib
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup

N_PER_BRANCH = 20
BRANCHES = ["dise", "myjk", "jibing", "nkwk", "fuke", "nxjk"]
ROOT = Path("/kaggle/working")
OUT = ROOT / "cnkang_review_v4"
OUT.mkdir(parents=True, exist_ok=True)

# URL lấy từ corpus, khác mẫu 120 URL trước.
FALLBACK_PATHS = [
    "dise/593/942688.html",
    "dise/241/501301.html",
    "dise/42/310579.html",
    "myjk/201207/433445.html",
    "myjk/201211/492236.html",
    "myjk/201112/415065.html",
    "jibing/hongyanbing/682910.html",
    "jibing/waieryan/205649.html",
    "jibing/azbzy/251682.html",
    "nkwk/manxingzhiqiguanyan/267031.html",
    "nkwk/gangdouyan/390150.html",
    "nkwk/IGAshenyan/361119.html",
    "fuke/ndjzl/589728.html",
    "fuke/yuejingbutiao/17030.html",
    "fuke/liuchanhou/19937.html",
    "nxjk/201212/783652.html",
    "nxjk/201109/562007.html",
    "nxjk/201109/563201.html",
]

def canonical(url):
    """Bỏ khác biệt HTTP/HTTPS và www khi kiểm tra URL đã test."""
    p = urlparse(str(url))
    host = (p.hostname or "").lower().rstrip(".")
    if host.startswith("www."):
        host = host[4:]
    return host + p.path + ("?" + p.query if p.query else "")

def host_ok(url):
    try:
        host = (urlparse(str(url)).hostname or "").lower().rstrip(".")
        return host == "cnkang.com" or host.endswith(".cnkang.com")
    except ValueError:
        return False

def compact(text):
    return re.sub(r"\s+", "", text)

# Đọc lịch sử: chỉ dùng để loại URL cũ, không tái sử dụng HTML.
seen = set()
for folder in sorted(ROOT.glob("cnkang_review*")):
    for filename in ["review.jsonl", "samples.jsonl"]:
        path = folder / filename
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                try:
                    item = json.loads(line)
                    url = item.get("source_url") or item.get("url")
                    if url:
                        seen.add(canonical(url))
                except (ValueError, TypeError):
                    pass

# Tái dựng mẫu 120 URL cũ theo cùng cách lấy mẫu seed=42,
# để vẫn loại được khi file lịch sử không còn nhưng dataset vẫn có.
try:
    files = list(Path("/kaggle/input").rglob("links_corpus_vi_en_zh.parquet"))
    if len(files) != 1:
        raise RuntimeError(f"Tìm thấy {len(files)} file Parquet phù hợp")

    df = pd.read_parquet(files[0], columns=["url"]).dropna().drop_duplicates("url")
    df = df[df.url.map(host_ok)].copy()
    df["branch"] = df.url.map(lambda u: urlparse(u).path.strip("/").split("/")[0])
    df = df[df.url.map(lambda u: urlparse(u).path.lower().endswith(".html"))]

    for branch in BRANCHES:
        group = df[df.branch == branch]
        seen.update(
            group.sample(min(20, len(group)), random_state=42).url.map(canonical)
        )

    df["canonical"] = df.url.map(canonical)
    fresh = df[~df.canonical.isin(seen)].drop_duplicates("canonical")
    URLS = []
    for branch in BRANCHES:
        group = fresh[fresh.branch == branch]
        URLS += group.sample(
            min(N_PER_BRANCH, len(group)), random_state=20261019
        ).url.tolist()
    if not URLS:
        raise RuntimeError("Không còn URL mới phù hợp")
    print("Dataset:", files[0])

except Exception as e:
    print("Dùng URL dự phòng:", e)
    URLS = [
        "http://www.cnkang.com/" + p for p in FALLBACK_PATHS
        if canonical("http://www.cnkang.com/" + p) not in seen
    ]

assert all(canonical(u) not in seen for u in URLS)
if not URLS:
    raise RuntimeError("Các URL dự phòng cũng đã test; cần dataset để chọn mẫu mới.")

print(f"Đã loại {len(seen):,} URL trong lịch sử/mẫu cũ.")
print(f"Sẽ tải {len(URLS)} URL mới.")

session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (compatible; CrawlDiagnostic/1.0)"
results = []

DATE_RE = re.compile(r"(?:19|20)\d{2}[-/.年]\d{1,2}[-/.月]\d{1,2}日?")
NOISE = [
    "核心提示", "您的位置", "手机查看", "扫描二维码", "凑数的文字",
    "相关文章", "免费提问", "立即咨询", "上一篇", "下一篇",
    "精彩推荐", "最新文章排行", "月排行", "总排行",
    "医学药学", "专题推荐", "推荐专题",
]
TOPICS = ["乳腺癌", "卵巢癌", "糖尿病", "淋巴瘤", "子宫内膜异位症"]

def render_text(node):
    # Tách đoạn theo block, không chèn khoảng trắng giữa các link trong câu.
    clone = BeautifulSoup(str(node), "html.parser")
    for tag in clone.select("script,style,nav,footer,aside,form"):
        tag.decompose()
    for tag in clone.find_all(["p", "div", "li", "h1", "h2", "h3", "tr", "br"]):
        tag.insert_before("\n")
        tag.insert_after("\n")
    return "\n".join(
        line for line in (
            re.sub(r"[ \t\u3000]+", " ", s).strip()
            for s in clone.get_text("", strip=False).splitlines()
        ) if line
    )

def clean_summary(text):
    """
    Chỉ bỏ khối đầu có 核心提示 khi:
    - tìm thấy dấu cắt tóm tắt trong tối đa 600 ký tự;
    - phần sau lặp lại đoạn mở đầu của tóm tắt.
    Không tìm đủ bằng chứng thì giữ nguyên và gắn cờ.
    """
    lines = text.splitlines()
    actions = []

    if lines and re.match(r"^\s*核心提示\s*[:：]", lines[0]):
        prefix = "\n".join(lines)
        cut = re.search(r"…+|\.{3,}", prefix[:600])
        if cut:
            label_end = prefix.find("\n")
            if label_end >= 0 and label_end < cut.start():
                summary = compact(prefix[label_end:cut.start()])
                remainder = prefix[cut.end():].lstrip()
                anchor = summary[:30]
                if (
                    len(anchor) >= 20
                    and anchor in compact(remainder)[:200]
                ):
                    actions.append({
                        "action": "remove_repeated_summary",
                        "removed": prefix[:cut.end()]
                    })
                    text = remainder

    # Chỉ bỏ các dòng giao diện độc lập, không cắt toàn bộ đuôi bài.
    standalone = {
        "相关文章", "免费提问", "立即咨询", "精彩推荐", "最新文章排行",
        "搞笑", "女性", "生活", "热点", "手机查看",
    }
    kept = []
    for line in text.splitlines():
        if line.strip() in standalone:
            actions.append({"action": "remove_ui_line", "removed": line})
        else:
            kept.append(line)
    return "\n".join(kept).strip(), actions

def order_signals(text):
    # Nhận cả “标题（五）…” và “5 标题…”.
    # Lưu riêng từng kiểu đánh số để giảm nhầm giữa cấp mục.
    sequences = {}
    patterns = {
        "chinese_brackets": r"[（(]([一二三四五六七八九十])[）)]",
        "arabic_brackets": r"[（(](\d{1,2})[）)]",
        "chinese_heading": r"^\s*([一二三四五六七八九十])[、．.]",
        "arabic_heading": r"^\s*(\d{1,2})(?:[、．.]|\s+)",
    }
    cn = dict(zip("一二三四五六七八九十", range(1, 11)))
    for name, pattern in patterns.items():
        values = []
        for line in text.splitlines():
            match = re.search(pattern, line[:100])
            if match:
                token = match.group(1)
                values.append(int(token) if token.isdigit() else cn[token])
        sequences[name] = values
    descending = any(
        values[i] > values[i+1] > values[i+2]
        for values in sequences.values()
        for i in range(len(values) - 2)
    )
    return descending, sequences

def body_candidates(soup):
    """Chỉ để review bố cục chưa biết, không coi ứng viên là selector đã đạt."""
    found = []
    for node in soup.select(
        '[itemprop="articleBody"], article, main, '
        '[id*="content"], [class*="content"], '
        '[id*="article"], [class*="article"]'
    ):
        text = node.get_text(" ", strip=True)
        n = len(compact(text))
        if not 100 <= n <= 25000:
            continue
        link_chars = sum(len(compact(a.get_text())) for a in node.find_all("a"))
        ratio = min(link_chars / max(n, 1), 1)
        score = n * (1 - ratio) ** 2 - sum(x in text for x in NOISE) * 1000
        found.append({
            "node": node.name,
            "id": node.get("id"),
            "class": node.get("class", []),
            "chars": n,
            "score": round(score, 1),
            "start": text[:160],
        })
    return sorted(found, key=lambda x: x["score"], reverse=True)[:3]

for index, source_url in enumerate(URLS, 1):
    branch = urlparse(source_url).path.strip("/").split("/")[0]
    key = hashlib.sha256(source_url.encode()).hexdigest()[:16]
    row = {
        "url": source_url, "branch": branch, "key": key,
        "status": None, "selector": "", "title": "", "text": "",
        "chars": 0, "flags": [], "date_candidates": [], "attempts": [],
    }

    try:
        response = None
        for target in dict.fromkeys([
            source_url.replace("http://", "https://", 1), source_url
        ]):
            time.sleep(1)
            try:
                response = session.get(target, timeout=(10, 25))
                row["attempts"].append({"url": target, "status": response.status_code})
                if response.status_code in (200, 403, 429):
                    break
            except requests.RequestException as e:
                row["attempts"].append({"url": target, "error": str(e)})

        if response is None:
            raise RuntimeError("Không kết nối được")
        row.update(status=response.status_code, final_url=response.url)
        response.raise_for_status()
        if not host_ok(response.url):
            raise RuntimeError("Chuyển hướng ra ngoài cnkang.com")

        (OUT / f"{key}.html").write_bytes(response.content)
        soup = BeautifulSoup(response.content, "html.parser")
        row["html_title"] = soup.title.get_text(" ", strip=True) if soup.title else ""

        # Selector đã có bằng chứng từ lần test trước.
        body = None
        selectors = [".detailc", "#endText"] if branch == "dise" else ["#endText", ".detailc"]
        for selector in selectors:
            matches = soup.select(selector)
            if len(matches) == 1 and len(compact(matches[0].get_text())) >= 40:
                body, row["selector"] = matches[0], selector
                break

        heading = soup.select_one(".detail h1") if body is not None and row["selector"] == ".detailc" else None
        heading = heading or soup.find("h1")
        row["title"] = (
            heading.get_text(" ", strip=True) if heading
            else row["html_title"].split("_")[0].strip()
        )

        if body is not None:
            raw_text = render_text(body)

            scope = body.find_parent(class_="detail") or body.find_parent(id="wleft")
            if scope is not None:
                whole = compact(scope.get_text(" ", strip=True))
                content = compact(body.get_text(" ", strip=True))
                position = whole.find(content)
                if position >= 0:
                    prefix = whole[:position][-600:]
                    row["date_candidates"] = list(dict.fromkeys(DATE_RE.findall(prefix)))
                    row["date_context"] = prefix[-250:]
        else:
            row["selector"] = "fallback"
            row["flags"].append("UNKNOWN_LAYOUT")
            row["body_candidates"] = body_candidates(soup)
            raw_text = trafilatura.extract(
                str(soup), url=response.url, include_comments=False,
                include_tables=False, favor_precision=True
            ) or ""

        text, actions = clean_summary(raw_text)
        row.update(text=text, chars=len(text), cleaning_actions=actions)
        row["noise_markers"] = [x for x in NOISE if x in text]

        if row["noise_markers"]:
            row["flags"].append("REMAINING_NOISE")
        if not text:
            row["flags"].append("EMPTY_TEXT")
        elif len(compact(text)) < 200:
            row["flags"].append("SHORT_TEXT_REVIEW")
        if not row["date_candidates"]:
            row["flags"].append("DATE_MISSING")
        if "\ufffd" in text:
            row["flags"].append("ENCODING_REVIEW")

        descending, sequences = order_signals(text)
        row["number_sequences"] = sequences
        if descending:
            row["flags"].append("DESCENDING_ORDER_REVIEW")

        absent = [t for t in TOPICS if t in row["title"] and t not in compact(text)]
        if absent:
            row["flags"].append("TITLE_TOPIC_REVIEW")
            row["missing_title_topics"] = absent

        paragraphs = [compact(p) for p in text.splitlines() if len(compact(p)) >= 40]
        row["duplicate_paragraphs"] = sum(v - 1 for v in Counter(paragraphs).values())
        if row["duplicate_paragraphs"]:
            row["flags"].append("DUPLICATE_PARAGRAPH_REVIEW")

        row["hash"] = hashlib.sha256(compact(text).encode()).hexdigest() if text else None
        (OUT / f"{key}_raw.txt").write_text(raw_text, encoding="utf-8")
        (OUT / f"{key}_clean.txt").write_text(text, encoding="utf-8")

    except Exception as e:
        row["error"] = str(e)
        row["flags"].append("FETCH_OR_PARSE_ERROR")

    results.append(row)
    print(
        f"[{index}/{len(URLS)}] /{branch}/ | HTTP {row['status']}"
        f" | {row['selector']} | {row['chars']} ký tự | {row['flags']}"
    )

# Trùng toàn văn giữa những URL khác nhau.
hashes = Counter(r.get("hash") for r in results if r.get("hash"))
for row in results:
    if row.get("hash") and hashes[row["hash"]] > 1:
        row["flags"].append("DUPLICATE_DOCUMENT_REVIEW")

(OUT / "review.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8"
)
pd.DataFrame([{
    "url": r["url"], "branch": r["branch"], "status": r["status"],
    "selector": r["selector"], "title": r["title"], "chars": r["chars"],
    "dates": "; ".join(r["date_candidates"]),
    "flags": "; ".join(r["flags"]),
} for r in results]).to_csv(OUT / "review.csv", index=False, encoding="utf-8-sig")

print("\n========== TỔNG KẾT ==========")
for branch in BRANCHES:
    items = [r for r in results if r["branch"] == branch]
    print(
        f"/{branch}/ | mẫu={len(items)}"
        f" | lỗi={sum('FETCH_OR_PARSE_ERROR' in r['flags'] for r in items)}"
        f" | chưa biết bố cục={sum('UNKNOWN_LAYOUT' in r['flags'] for r in items)}"
        f" | bỏ tóm tắt lặp={sum(any(a['action']=='remove_repeated_summary' for a in r.get('cleaning_actions', [])) for r in items)}"
        f" | còn nhiễu={sum('REMAINING_NOISE' in r['flags'] for r in items)}"
        f" | cần xem thứ tự={sum('DESCENDING_ORDER_REVIEW' in r['flags'] for r in items)}"
        f" | có ngày={sum(bool(r['date_candidates']) for r in items)}"
    )

print("\n========== MẪU MỖI NHÁNH ==========")
for branch in BRANCHES:
    row = next((r for r in results if r["branch"] == branch and r["text"]), None)
    if row is None:
        continue
    print("\n" + "=" * 75)
    print(row["url"])
    print("TITLE:", row["title"])
    print("SELECTOR:", row["selector"], "| DATE:", row["date_candidates"])
    print("FLAGS:", row["flags"])
    print("SỐ THỨ TỰ:", row.get("number_sequences"))
    print("ĐẦU:\n", row["text"][:650])
    print("CUỐI:\n", row["text"][-300:])

print("\n========== BỐ CỤC CHƯA XÁC ĐỊNH ==========")
for row in results:
    if "UNKNOWN_LAYOUT" in row["flags"]:
        print(row["url"])
        print(json.dumps(row.get("body_candidates", []), ensure_ascii=False, indent=2))

print("\nĐã lưu:", OUT)
print("Không tự đảo đoạn hoặc loại bài ngắn.")
print("Ngày và cờ chất lượng vẫn là tín hiệu cần review, chưa phải xác nhận nội dung đúng.")

Note: you may need to restart the kernel to use updated packages.
Dataset: /kaggle/input/datasets/lighth/r2ai-stage3/links_corpus_vi_en_zh.parquet
Đã loại 120 URL trong lịch sử/mẫu cũ.
Sẽ tải 120 URL mới.
[1/120] /dise/ | HTTP 200 | .detailc | 1375 ký tự | []
[2/120] /dise/ | HTTP 200 | .detailc | 225 ký tự | ['TITLE_TOPIC_REVIEW']
[3/120] /dise/ | HTTP 200 | .detailc | 1038 ký tự | []
[4/120] /dise/ | HTTP 200 | .detailc | 947 ký tự | []
[5/120] /dise/ | HTTP 200 | .detailc | 624 ký tự | []
[6/120] /dise/ | HTTP 200 | .detailc | 519 ký tự | []
[7/120] /dise/ | HTTP 200 | .detailc | 464 ký tự | []
[8/120] /dise/ | HTTP 200 | .detailc | 2232 ký tự | []
[9/120] /dise/ | HTTP 200 | .detailc | 1884 ký tự | []
[10/120] /dise/ | HTTP 200 | .detailc | 1040 ký tự | []
[11/120] /dise/ | HTTP 200 | .detailc | 946 ký tự | []
[12/120] /dise/ | HTTP 200 | .detailc | 1315 ký tự | []
[13/120] /dise/ | HTTP 200 | .detailc | 12987 ký tự | ['DESCENDING_ORDER_REVIEW']
[14/120] /dise/ | HTTP 200 | .detail

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


[111/120] /nxjk/ | HTTP 200 | #endText | 1539 ký tự | ['REMAINING_NOISE', 'ENCODING_REVIEW']
[112/120] /nxjk/ | HTTP 200 | fallback | 309 ký tự | ['UNKNOWN_LAYOUT', 'DATE_MISSING']
[113/120] /nxjk/ | HTTP 200 | fallback | 268 ký tự | ['UNKNOWN_LAYOUT', 'DATE_MISSING']
[114/120] /nxjk/ | HTTP 200 | fallback | 618 ký tự | ['UNKNOWN_LAYOUT', 'DATE_MISSING']
[115/120] /nxjk/ | HTTP 200 | #endText | 776 ký tự | ['REMAINING_NOISE']
[116/120] /nxjk/ | HTTP 200 | fallback | 821 ký tự | ['UNKNOWN_LAYOUT', 'DATE_MISSING']
[117/120] /nxjk/ | HTTP 200 | #endText | 484 ký tự | ['REMAINING_NOISE']
[118/120] /nxjk/ | HTTP 200 | fallback | 1598 ký tự | ['UNKNOWN_LAYOUT', 'DATE_MISSING']
[119/120] /nxjk/ | HTTP 200 | fallback | 1034 ký tự | ['UNKNOWN_LAYOUT', 'DATE_MISSING']
[120/120] /nxjk/ | HTTP 200 | #endText | 666 ký tự | ['REMAINING_NOISE']

========== TỔNG KẾT ==========
/dise/ | mẫu=20 | lỗi=0 | chưa biết bố cục=0 | bỏ tóm tắt lặp=0 | còn nhiễu=0 | cần xem thứ tự=1 | có ngày=20
/myjk/ | mẫu=20 

In [14]:
%pip -q install beautifulsoup4 trafilatura

import re, json, time, hashlib
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup

N_PER_BRANCH = 20
BRANCHES = ["dise", "myjk", "jibing", "nkwk", "fuke", "nxjk"]
ROOT = Path("/kaggle/working")
OUT = ROOT / "cnkang_review_v4"
OUT.mkdir(parents=True, exist_ok=True)

# URL lấy từ corpus, khác mẫu 120 URL trước.
FALLBACK_PATHS = [
    "dise/593/942688.html",
    "dise/241/501301.html",
    "dise/42/310579.html",
    "myjk/201207/433445.html",
    "myjk/201211/492236.html",
    "myjk/201112/415065.html",
    "jibing/hongyanbing/682910.html",
    "jibing/waieryan/205649.html",
    "jibing/azbzy/251682.html",
    "nkwk/manxingzhiqiguanyan/267031.html",
    "nkwk/gangdouyan/390150.html",
    "nkwk/IGAshenyan/361119.html",
    "fuke/ndjzl/589728.html",
    "fuke/yuejingbutiao/17030.html",
    "fuke/liuchanhou/19937.html",
    "nxjk/201212/783652.html",
    "nxjk/201109/562007.html",
    "nxjk/201109/563201.html",
]

def canonical(url):
    """Bỏ khác biệt HTTP/HTTPS và www khi kiểm tra URL đã test."""
    p = urlparse(str(url))
    host = (p.hostname or "").lower().rstrip(".")
    if host.startswith("www."):
        host = host[4:]
    return host + p.path + ("?" + p.query if p.query else "")

def host_ok(url):
    try:
        host = (urlparse(str(url)).hostname or "").lower().rstrip(".")
        return host == "cnkang.com" or host.endswith(".cnkang.com")
    except ValueError:
        return False

def compact(text):
    return re.sub(r"\s+", "", text)

# Đọc lịch sử: chỉ dùng để loại URL cũ, không tái sử dụng HTML.
seen = set()
for folder in sorted(ROOT.glob("cnkang_review*")):
    for filename in ["review.jsonl", "samples.jsonl"]:
        path = folder / filename
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                try:
                    item = json.loads(line)
                    url = item.get("source_url") or item.get("url")
                    if url:
                        seen.add(canonical(url))
                except (ValueError, TypeError):
                    pass

# Tái dựng mẫu 120 URL cũ theo cùng cách lấy mẫu seed=42,
# để vẫn loại được khi file lịch sử không còn nhưng dataset vẫn có.
try:
    files = list(Path("/kaggle/input").rglob("links_corpus_vi_en_zh.parquet"))
    if len(files) != 1:
        raise RuntimeError(f"Tìm thấy {len(files)} file Parquet phù hợp")

    df = pd.read_parquet(files[0], columns=["url"]).dropna().drop_duplicates("url")
    df = df[df.url.map(host_ok)].copy()
    df["branch"] = df.url.map(lambda u: urlparse(u).path.strip("/").split("/")[0])
    df = df[df.url.map(lambda u: urlparse(u).path.lower().endswith(".html"))]

    for branch in BRANCHES:
        group = df[df.branch == branch]
        seen.update(
            group.sample(min(20, len(group)), random_state=42).url.map(canonical)
        )

    df["canonical"] = df.url.map(canonical)
    fresh = df[~df.canonical.isin(seen)].drop_duplicates("canonical")
    URLS = []
    for branch in BRANCHES:
        group = fresh[fresh.branch == branch]
        URLS += group.sample(
            min(N_PER_BRANCH, len(group)), random_state=20261019
        ).url.tolist()
    if not URLS:
        raise RuntimeError("Không còn URL mới phù hợp")
    print("Dataset:", files[0])

except Exception as e:
    print("Dùng URL dự phòng:", e)
    URLS = [
        "http://www.cnkang.com/" + p for p in FALLBACK_PATHS
        if canonical("http://www.cnkang.com/" + p) not in seen
    ]

assert all(canonical(u) not in seen for u in URLS)
if not URLS:
    raise RuntimeError("Các URL dự phòng cũng đã test; cần dataset để chọn mẫu mới.")

print(f"Đã loại {len(seen):,} URL trong lịch sử/mẫu cũ.")
print(f"Sẽ tải {len(URLS)} URL mới.")

session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (compatible; CrawlDiagnostic/1.0)"
results = []

DATE_RE = re.compile(r"(?:19|20)\d{2}[-/.年]\d{1,2}[-/.月]\d{1,2}日?")
NOISE = [
    "核心提示", "您的位置", "手机查看", "扫描二维码", "凑数的文字",
    "相关文章", "免费提问", "立即咨询", "上一篇", "下一篇",
    "精彩推荐", "最新文章排行", "月排行", "总排行",
    "医学药学", "专题推荐", "推荐专题",
]
TOPICS = ["乳腺癌", "卵巢癌", "糖尿病", "淋巴瘤", "子宫内膜异位症"]

def render_text(node):
    # Tách đoạn theo block, không chèn khoảng trắng giữa các link trong câu.
    clone = BeautifulSoup(str(node), "html.parser")
    for tag in clone.select("script,style,nav,footer,aside,form"):
        tag.decompose()
    for tag in clone.find_all(["p", "div", "li", "h1", "h2", "h3", "tr", "br"]):
        tag.insert_before("\n")
        tag.insert_after("\n")
    return "\n".join(
        line for line in (
            re.sub(r"[ \t\u3000]+", " ", s).strip()
            for s in clone.get_text("", strip=False).splitlines()
        ) if line
    )

def clean_summary(text):
    """
    Chỉ bỏ khối đầu có 核心提示 khi:
    - tìm thấy dấu cắt tóm tắt trong tối đa 600 ký tự;
    - phần sau lặp lại đoạn mở đầu của tóm tắt.
    Không tìm đủ bằng chứng thì giữ nguyên và gắn cờ.
    """
    lines = text.splitlines()
    actions = []

    if lines and re.match(r"^\s*核心提示\s*[:：]", lines[0]):
        prefix = "\n".join(lines)
        cut = re.search(r"…+|\.{3,}", prefix[:600])
        if cut:
            label_end = prefix.find("\n")
            if label_end >= 0 and label_end < cut.start():
                summary = compact(prefix[label_end:cut.start()])
                remainder = prefix[cut.end():].lstrip()
                anchor = summary[:30]
                if (
                    len(anchor) >= 20
                    and anchor in compact(remainder)[:200]
                ):
                    actions.append({
                        "action": "remove_repeated_summary",
                        "removed": prefix[:cut.end()]
                    })
                    text = remainder

    # Chỉ bỏ các dòng giao diện độc lập, không cắt toàn bộ đuôi bài.
    standalone = {
        "相关文章", "免费提问", "立即咨询", "精彩推荐", "最新文章排行",
        "搞笑", "女性", "生活", "热点", "手机查看",
    }
    kept = []
    for line in text.splitlines():
        if line.strip() in standalone:
            actions.append({"action": "remove_ui_line", "removed": line})
        else:
            kept.append(line)
    return "\n".join(kept).strip(), actions

def order_signals(text):
    # Nhận cả “标题（五）…” và “5 标题…”.
    # Lưu riêng từng kiểu đánh số để giảm nhầm giữa cấp mục.
    sequences = {}
    patterns = {
        "chinese_brackets": r"[（(]([一二三四五六七八九十])[）)]",
        "arabic_brackets": r"[（(](\d{1,2})[）)]",
        "chinese_heading": r"^\s*([一二三四五六七八九十])[、．.]",
        "arabic_heading": r"^\s*(\d{1,2})(?:[、．.]|\s+)",
    }
    cn = dict(zip("一二三四五六七八九十", range(1, 11)))
    for name, pattern in patterns.items():
        values = []
        for line in text.splitlines():
            match = re.search(pattern, line[:100])
            if match:
                token = match.group(1)
                values.append(int(token) if token.isdigit() else cn[token])
        sequences[name] = values
    descending = any(
        values[i] > values[i+1] > values[i+2]
        for values in sequences.values()
        for i in range(len(values) - 2)
    )
    return descending, sequences

def body_candidates(soup):
    """Chỉ để review bố cục chưa biết, không coi ứng viên là selector đã đạt."""
    found = []
    for node in soup.select(
        '[itemprop="articleBody"], article, main, '
        '[id*="content"], [class*="content"], '
        '[id*="article"], [class*="article"]'
    ):
        text = node.get_text(" ", strip=True)
        n = len(compact(text))
        if not 100 <= n <= 25000:
            continue
        link_chars = sum(len(compact(a.get_text())) for a in node.find_all("a"))
        ratio = min(link_chars / max(n, 1), 1)
        score = n * (1 - ratio) ** 2 - sum(x in text for x in NOISE) * 1000
        found.append({
            "node": node.name,
            "id": node.get("id"),
            "class": node.get("class", []),
            "chars": n,
            "score": round(score, 1),
            "start": text[:160],
        })
    return sorted(found, key=lambda x: x["score"], reverse=True)[:3]

for index, source_url in enumerate(URLS, 1):
    branch = urlparse(source_url).path.strip("/").split("/")[0]
    key = hashlib.sha256(source_url.encode()).hexdigest()[:16]
    row = {
        "url": source_url, "branch": branch, "key": key,
        "status": None, "selector": "", "title": "", "text": "",
        "chars": 0, "flags": [], "date_candidates": [], "attempts": [],
    }

    try:
        response = None
        for target in dict.fromkeys([
            source_url.replace("http://", "https://", 1), source_url
        ]):
            time.sleep(1)
            try:
                response = session.get(target, timeout=(10, 25))
                row["attempts"].append({"url": target, "status": response.status_code})
                if response.status_code in (200, 403, 429):
                    break
            except requests.RequestException as e:
                row["attempts"].append({"url": target, "error": str(e)})

        if response is None:
            raise RuntimeError("Không kết nối được")
        row.update(status=response.status_code, final_url=response.url)
        response.raise_for_status()
        if not host_ok(response.url):
            raise RuntimeError("Chuyển hướng ra ngoài cnkang.com")

        (OUT / f"{key}.html").write_bytes(response.content)
        soup = BeautifulSoup(response.content, "html.parser")
        row["html_title"] = soup.title.get_text(" ", strip=True) if soup.title else ""

        # Selector đã có bằng chứng từ lần test trước.
        body = None
        selectors = [".detailc", "#endText"] if branch == "dise" else ["#endText", ".detailc"]
        for selector in selectors:
            matches = soup.select(selector)
            if len(matches) == 1 and len(compact(matches[0].get_text())) >= 40:
                body, row["selector"] = matches[0], selector
                break

        heading = soup.select_one(".detail h1") if body is not None and row["selector"] == ".detailc" else None
        heading = heading or soup.find("h1")
        row["title"] = (
            heading.get_text(" ", strip=True) if heading
            else row["html_title"].split("_")[0].strip()
        )

        if body is not None:
            raw_text = render_text(body)

            scope = body.find_parent(class_="detail") or body.find_parent(id="wleft")
            if scope is not None:
                whole = compact(scope.get_text(" ", strip=True))
                content = compact(body.get_text(" ", strip=True))
                position = whole.find(content)
                if position >= 0:
                    prefix = whole[:position][-600:]
                    row["date_candidates"] = list(dict.fromkeys(DATE_RE.findall(prefix)))
                    row["date_context"] = prefix[-250:]
        else:
            row["selector"] = "fallback"
            row["flags"].append("UNKNOWN_LAYOUT")
            row["body_candidates"] = body_candidates(soup)
            raw_text = trafilatura.extract(
                str(soup), url=response.url, include_comments=False,
                include_tables=False, favor_precision=True
            ) or ""

        text, actions = clean_summary(raw_text)
        row.update(text=text, chars=len(text), cleaning_actions=actions)
        row["noise_markers"] = [x for x in NOISE if x in text]

        if row["noise_markers"]:
            row["flags"].append("REMAINING_NOISE")
        if not text:
            row["flags"].append("EMPTY_TEXT")
        elif len(compact(text)) < 200:
            row["flags"].append("SHORT_TEXT_REVIEW")
        if not row["date_candidates"]:
            row["flags"].append("DATE_MISSING")
        if "\ufffd" in text:
            row["flags"].append("ENCODING_REVIEW")

        descending, sequences = order_signals(text)
        row["number_sequences"] = sequences
        if descending:
            row["flags"].append("DESCENDING_ORDER_REVIEW")

        absent = [t for t in TOPICS if t in row["title"] and t not in compact(text)]
        if absent:
            row["flags"].append("TITLE_TOPIC_REVIEW")
            row["missing_title_topics"] = absent

        paragraphs = [compact(p) for p in text.splitlines() if len(compact(p)) >= 40]
        row["duplicate_paragraphs"] = sum(v - 1 for v in Counter(paragraphs).values())
        if row["duplicate_paragraphs"]:
            row["flags"].append("DUPLICATE_PARAGRAPH_REVIEW")

        row["hash"] = hashlib.sha256(compact(text).encode()).hexdigest() if text else None
        (OUT / f"{key}_raw.txt").write_text(raw_text, encoding="utf-8")
        (OUT / f"{key}_clean.txt").write_text(text, encoding="utf-8")

    except Exception as e:
        row["error"] = str(e)
        row["flags"].append("FETCH_OR_PARSE_ERROR")

    results.append(row)
    print(
        f"[{index}/{len(URLS)}] /{branch}/ | HTTP {row['status']}"
        f" | {row['selector']} | {row['chars']} ký tự | {row['flags']}"
    )

# Trùng toàn văn giữa những URL khác nhau.
hashes = Counter(r.get("hash") for r in results if r.get("hash"))
for row in results:
    if row.get("hash") and hashes[row["hash"]] > 1:
        row["flags"].append("DUPLICATE_DOCUMENT_REVIEW")

(OUT / "review.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8"
)
pd.DataFrame([{
    "url": r["url"], "branch": r["branch"], "status": r["status"],
    "selector": r["selector"], "title": r["title"], "chars": r["chars"],
    "dates": "; ".join(r["date_candidates"]),
    "flags": "; ".join(r["flags"]),
} for r in results]).to_csv(OUT / "review.csv", index=False, encoding="utf-8-sig")

print("\n========== TỔNG KẾT ==========")
for branch in BRANCHES:
    items = [r for r in results if r["branch"] == branch]
    print(
        f"/{branch}/ | mẫu={len(items)}"
        f" | lỗi={sum('FETCH_OR_PARSE_ERROR' in r['flags'] for r in items)}"
        f" | chưa biết bố cục={sum('UNKNOWN_LAYOUT' in r['flags'] for r in items)}"
        f" | bỏ tóm tắt lặp={sum(any(a['action']=='remove_repeated_summary' for a in r.get('cleaning_actions', [])) for r in items)}"
        f" | còn nhiễu={sum('REMAINING_NOISE' in r['flags'] for r in items)}"
        f" | cần xem thứ tự={sum('DESCENDING_ORDER_REVIEW' in r['flags'] for r in items)}"
        f" | có ngày={sum(bool(r['date_candidates']) for r in items)}"
    )

print("\n========== MẪU MỖI NHÁNH ==========")
for branch in BRANCHES:
    row = next((r for r in results if r["branch"] == branch and r["text"]), None)
    if row is None:
        continue
    print("\n" + "=" * 75)
    print(row["url"])
    print("TITLE:", row["title"])
    print("SELECTOR:", row["selector"], "| DATE:", row["date_candidates"])
    print("FLAGS:", row["flags"])
    print("SỐ THỨ TỰ:", row.get("number_sequences"))
    print("ĐẦU:\n", row["text"][:650])
    print("CUỐI:\n", row["text"][-300:])

print("\n========== BỐ CỤC CHƯA XÁC ĐỊNH ==========")
for row in results:
    if "UNKNOWN_LAYOUT" in row["flags"]:
        print(row["url"])
        print(json.dumps(row.get("body_candidates", []), ensure_ascii=False, indent=2))

print("\nĐã lưu:", OUT)
print("Không tự đảo đoạn hoặc loại bài ngắn.")
print("Ngày và cờ chất lượng vẫn là tín hiệu cần review, chưa phải xác nhận nội dung đúng.")

Note: you may need to restart the kernel to use updated packages.
Dataset: /kaggle/input/datasets/lighth/r2ai-stage3/links_corpus_vi_en_zh.parquet
Đã loại 240 URL trong lịch sử/mẫu cũ.
Sẽ tải 120 URL mới.
[1/120] /dise/ | HTTP 200 | .detailc | 3126 ký tự | []
[2/120] /dise/ | HTTP 200 | .detailc | 1120 ký tự | []
[3/120] /dise/ | HTTP 200 | .detailc | 1113 ký tự | []
[4/120] /dise/ | HTTP 200 | .detailc | 693 ký tự | []
[5/120] /dise/ | HTTP 200 | .detailc | 1115 ký tự | ['DUPLICATE_PARAGRAPH_REVIEW']
[6/120] /dise/ | HTTP 200 | .detailc | 1055 ký tự | []
[7/120] /dise/ | HTTP 200 | .detailc | 708 ký tự | []
[8/120] /dise/ | HTTP 200 | .detailc | 792 ký tự | []
[9/120] /dise/ | HTTP 200 | .detailc | 399 ký tự | []
[10/120] /dise/ | HTTP 200 | .detailc | 734 ký tự | []
[11/120] /dise/ | HTTP 200 | .detailc | 514 ký tự | []
[12/120] /dise/ | HTTP 200 | .detailc | 710 ký tự | []
[13/120] /dise/ | HTTP 200 | .detailc | 687 ký tự | []
[14/120] /dise/ | HTTP 200 | .detailc | 12987 ký tự | ['

In [15]:
%pip -q install beautifulsoup4

import re, json, hashlib, difflib
from pathlib import Path
from collections import Counter
import pandas as pd
from bs4 import BeautifulSoup

SRC = Path("/kaggle/working/cnkang_review_v4")
OUT = Path("/kaggle/working/cnkang_review_v5")
OUT.mkdir(parents=True, exist_ok=True)

manifest = SRC / "review.jsonl"
if not manifest.exists():
    raise FileNotFoundError("Không thấy review_v4/review.jsonl. Chạy trong session Kaggle trước đó.")

old_rows = [
    json.loads(line) for line in manifest.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
results = []

def compact(text):
    return re.sub(r"\s+", "", text)

def cjk_ratio(text):
    meaningful = re.sub(r"\s+", "", text)
    return len(re.findall(r"[\u4e00-\u9fff]", meaningful)) / max(len(meaningful), 1)

def render(node):
    clone = BeautifulSoup(str(node), "html.parser")
    for tag in clone.select("script,style,nav,footer,aside,form"):
        tag.decompose()
    for tag in clone.find_all(["p", "div", "li", "h1", "h2", "h3", "tr", "br"]):
        tag.insert_before("\n")
        tag.insert_after("\n")
    return "\n".join(
        line for line in (
            re.sub(r"[ \t\u3000]+", " ", s).strip()
            for s in clone.get_text("", strip=False).splitlines()
        ) if line
    )

def decode_html(raw):
    head = raw[:16000].decode("latin1")
    match = re.search(r'charset\s*=\s*["\']?\s*([\w-]+)', head, re.I)
    declared = match.group(1).lower() if match else ""
    candidates = list(dict.fromkeys(
        [declared, "utf-8", "gb18030", "gbk", "big5"]
    ))
    scored = []
    for encoding in candidates:
        if not encoding:
            continue
        try:
            html = raw.decode(encoding, errors="strict")
        except (UnicodeError, LookupError):
            continue

        soup = BeautifulSoup(html, "html.parser")
        for tag in soup.select("script,style"):
            tag.decompose()
        text = soup.get_text(" ", strip=True)
        ratio = cjk_ratio(text)
        # Các ký tự thường gặp khi byte Trung bị giải mã thành chữ Latin.
        bad = len(re.findall(r"[ÃÂÕÈÊÐÖØËÄÎÏÒÓº¼½¾]", text)) / max(len(text), 1)
        score = ratio * 100 - bad * 150
        if encoding == declared:
            score += 5
        if encoding == "utf-8":
            score += 10
        scored.append((score, encoding, html, ratio))

    if not scored:
        return raw.decode("utf-8", errors="replace"), "utf-8-replace", [], True

    scored.sort(key=lambda x: x[0], reverse=True)
    best = scored[0]
    diagnostics = [
        {"encoding": enc, "score": round(score, 2), "cjk_ratio": round(ratio, 3)}
        for score, enc, _, ratio in scored
    ]
    uncertain = best[3] < 0.2
    return best[2], best[1], diagnostics, uncertain

NOISE = [
    "核心提示", "您的位置", "手机查看", "扫描二维码", "凑数的文字",
    "相关文章", "免费提问", "立即咨询", "上一篇", "下一篇",
    "精彩推荐", "最新文章排行", "月排行", "总排行",
    "医学药学", "专题推荐", "推荐专题",
]

def label(node):
    return (
        node.name + ("#" + node["id"] if node.get("id") else "")
        + ("." + ".".join(node.get("class", [])) if node.get("class") else "")
    )[:160]

def choose_body(soup):
    # Chấp nhận các selector đã có bằng chứng, kể cả khác chữ hoa/thường của ID.
    exact = soup.select(".detailc")
    exact += soup.find_all(id=re.compile(r"^endtext$", re.I))
    unique = []
    for node in exact:
        if not any(node is previous for previous in unique):
            unique.append(node)
    eligible = [n for n in unique if len(compact(n.get_text())) >= 40]
    if len(eligible) == 1:
        return eligible[0], label(eligible[0]), False, []

    # Bố cục chưa biết: xét cả table/td và mọi tên id/class.
    # Chỉ chọn ứng viên để review, không tự xác nhận là parser chuẩn.
    clean = BeautifulSoup(str(soup), "html.parser")
    for tag in clean.select("script,style,nav,header,footer,aside,form"):
        tag.decompose()
    candidates = []
    for node in clean.find_all(["article", "main", "div", "section", "td", "table"]):
        text = node.get_text(" ", strip=True)
        n = len(compact(text))
        if not 80 <= n <= 40000:
            continue
        links = sum(len(compact(a.get_text())) for a in node.find_all("a"))
        ratio = min(links / max(n, 1), 1)
        long_p = sum(len(compact(p.get_text())) >= 40 for p in node.find_all("p"))
        name = " ".join([str(node.get("id", "")), *node.get("class", [])])
        named_bonus = 150 if re.search(r"content|article|text|detail|正文", name, re.I) else 0
        score = (
            min(n, 15000) * (1 - ratio) ** 2
            + min(long_p, 20) * 60 + named_bonus
            - sum(x in text for x in NOISE) * 800
        )
        candidates.append((score, node, n, ratio))

    candidates.sort(key=lambda x: x[0], reverse=True)
    info = [{
        "node": label(node), "chars": n, "link_ratio": round(ratio, 3),
        "score": round(score, 1), "start": node.get_text(" ", strip=True)[:180],
    } for score, node, n, ratio in candidates[:5]]
    return (
        (candidates[0][1], label(candidates[0][1]), True, info)
        if candidates else (None, "", True, [])
    )

def remove_summary(text, title):
    """
    Bỏ phần tóm tắt khi tìm được:
    核心提示 + tóm tắt bị cắt + đoạn mở đầu lặp lại trong phần sau.
    So sánh trên văn bản bỏ khoảng trắng, cắt trên vị trí văn bản gốc.
    """
    actions = []
    chars = list(re.finditer(r"\S", text))
    normalized = "".join(m.group() for m in chars)
    marker = re.search(r"核心提示[:：]?", normalized[:100])
    if not marker:
        return text, actions

    start = marker.end()
    normalized_title = compact(title)
    if normalized_title and normalized.startswith(normalized_title, start):
        start += len(normalized_title)

    cutoff = re.search(r"…+|\.{3,}", normalized[start:start + 700])
    if not cutoff:
        return text, actions

    cut_start = start + cutoff.start()
    cut_end = start + cutoff.end()
    summary = normalized[start:cut_start]
    # Cần đủ dài để tránh cắt nhầm do vài chữ trùng nhau.
    if len(summary) < 20:
        return text, actions

    anchor = summary[:30]
    search_end = min(len(normalized), cut_end + 400)
    repeat = normalized.find(anchor, cut_end, search_end)

    # Dự phòng khác biệt nhỏ về dấu câu ở đầu đoạn.
    if repeat < 0:
        after = normalized[cut_end:search_end]
        match = difflib.SequenceMatcher(
            None, summary[:100], after, autojunk=False
        ).find_longest_match()
        if match.size >= 30 and match.a <= 5 and match.b <= 150:
            repeat = cut_end + match.b - match.a

    if repeat >= cut_end:
        original_pos = chars[repeat].start()
        actions.append({
            "action": "remove_repeated_summary",
            "removed": text[:original_pos],
            "evidence_anchor": anchor,
        })
        return text[original_pos:].lstrip(), actions

    return text, actions

def order_check(text):
    cn = dict(zip("一二三四五六七八九十", range(1, 11)))
    patterns = {
        "cn_brackets": r"[（(]([一二三四五六七八九十])[）)]",
        "num_brackets": r"[（(](\d{1,2})[）)]",
        "cn_heading": r"^\s*([一二三四五六七八九十])[、．.]",
        "num_heading": r"^\s*(\d{1,2})(?:[、．.]|\s+)",
    }
    sequences = {}
    for name, pattern in patterns.items():
        values = []
        for line in text.splitlines():
            m = re.search(pattern, line[:120])
            if m:
                token = m.group(1)
                values.append(int(token) if token.isdigit() else cn[token])
        sequences[name] = values
    descending = any(
        seq[i] > seq[i+1] > seq[i+2]
        for seq in sequences.values() for i in range(len(seq) - 2)
    )
    return descending, sequences

DATE_RE = re.compile(r"(?:19|20)\d{2}[-/.年]\d{1,2}[-/.月]\d{1,2}日?")
UI_LINES = {
    "相关文章", "免费提问", "立即咨询", "精彩推荐", "最新文章排行",
    "搞笑", "女性", "生活", "热点", "手机查看",
}

for i, old in enumerate(old_rows, 1):
    key = old["key"]
    row = {
        "url": old["url"], "branch": old["branch"], "key": key,
        "flags": [], "source_flags": [], "text": "", "chars": 0,
        "date_candidates": [], "selector": "", "summary_removed": False,
    }
    try:
        path = SRC / f"{key}.html"
        if not path.exists():
            raise FileNotFoundError("Không có HTML cache")
        raw = path.read_bytes()
        html, encoding, diagnostics, uncertain = decode_html(raw)
        row.update(encoding=encoding, encoding_candidates=diagnostics)
        if uncertain:
            row["flags"].append("ENCODING_UNCERTAIN")

        soup = BeautifulSoup(html, "html.parser")
        row["html_title"] = soup.title.get_text(" ", strip=True) if soup.title else ""
        heading = soup.select_one(".detail h1") or soup.find("h1")
        title = (
            heading.get_text(" ", strip=True) if heading
            else row["html_title"].split("_")[0].strip()
        )
        row["title"] = title

        body, selector, guessed, candidates = choose_body(soup)
        row.update(selector=selector, body_candidates=candidates)
        if guessed:
            row["flags"].append("GUESSED_LAYOUT_REVIEW")
        if body is None:
            raise ValueError("Chưa tìm được vùng nội dung")

        before = render(body)
        text, actions = remove_summary(before, title)
        row["summary_removed"] = bool(actions)
        row["cleaning_actions"] = actions
        text = "\n".join(s for s in text.splitlines() if s.strip() not in UI_LINES).strip()

        row.update(text=text, chars=len(text), body_cjk_ratio=round(cjk_ratio(text), 3))
        if cjk_ratio(text) < 0.2 or "\ufffd" in text:
            row["flags"].append("BODY_ENCODING_REVIEW")
        if not text:
            row["flags"].append("EMPTY_TEXT")
        elif len(compact(text)) < 200:
            row["flags"].append("SHORT_TEXT_REVIEW")

        row["noise_markers"] = [x for x in NOISE if x in text]
        if row["noise_markers"]:
            row["flags"].append("REMAINING_NOISE")

        descending, sequences = order_check(text)
        row["number_sequences"] = sequences
        if descending:
            row["flags"].append("DESCENDING_ORDER_REVIEW")

        paragraphs = [compact(p) for p in text.splitlines() if len(compact(p)) >= 40]
        if any(count > 1 for count in Counter(paragraphs).values()):
            row["flags"].append("DUPLICATE_PARAGRAPH_REVIEW")

        # Ngày: chỉ tìm trong vùng đứng trước phần thân bài.
        scope = body.find_parent(class_="detail") or body.find_parent(id="wleft")
        if scope is not None:
            all_text = compact(scope.get_text(" ", strip=True))
            body_text = compact(body.get_text(" ", strip=True))
            position = all_text.find(body_text)
            if position >= 0:
                context = all_text[:position][-600:]
                row["date_candidates"] = list(dict.fromkeys(DATE_RE.findall(context)))
                row["date_context"] = context[-250:]
        if not row["date_candidates"]:
            row["flags"].append("DATE_MISSING")

        # Tín hiệu chất lượng nguồn, không tự kết luận đúng/sai y khoa.
        source_patterns = {
            "PROMOTIONAL_CLAIM_REVIEW":
                r"无毒副作用|无任何副作用|纯天然绿色饮品|包治|保证治愈",
            "PRODUCT_PROMOTION_REVIEW":
                r"长期饮用.{0,25}茶|万能膏|购买|订购|咨询热线",
            "PERSONAL_EXPERIENCE_REVIEW":
                r"笔者|帅哥一枚|我就发现|我的经验|我自己",
        }
        row["source_evidence"] = {}
        for flag, pattern in source_patterns.items():
            matches = list(re.finditer(pattern, text))
            if matches:
                row["source_flags"].append(flag)
                row["source_evidence"][flag] = [
                    text[max(0, m.start()-35):m.end()+60] for m in matches[:3]
                ]

        # Giữ phép kiểm tra lệch chủ đề đã có ở v4.
        for topic in ["乳腺癌", "卵巢癌", "糖尿病", "淋巴瘤", "子宫内膜异位症"]:
            if topic in title and topic not in compact(text):
                row["flags"].append("TITLE_TOPIC_REVIEW")
                break

        row["old_chars"] = old.get("chars", 0)
        row["old_title"] = old.get("title", "")
        row["hash"] = hashlib.sha256(compact(text).encode()).hexdigest() if text else None
        (OUT / f"{key}_before.txt").write_text(before, encoding="utf-8")
        (OUT / f"{key}_clean.txt").write_text(text, encoding="utf-8")

    except Exception as e:
        row["flags"].append("PARSE_ERROR")
        row["error"] = str(e)

    results.append(row)
    print(f"[{i}/{len(old_rows)}] /{row['branch']}/ | "
          f"{row.get('encoding', '?')} | {row['selector']} | "
          f"{row['chars']} ký tự | {row['flags']} | nguồn={row['source_flags']}")

hashes = Counter(r.get("hash") for r in results if r.get("hash"))
for row in results:
    if row.get("hash") and hashes[row["hash"]] > 1:
        row["flags"].append("DUPLICATE_DOCUMENT_REVIEW")

(OUT / "review.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8"
)
pd.DataFrame([{
    "url": r["url"], "branch": r["branch"],
    "encoding": r.get("encoding"), "selector": r["selector"],
    "title": r.get("title", ""), "chars": r["chars"],
    "summary_removed": r["summary_removed"],
    "flags": "; ".join(r["flags"]),
    "source_flags": "; ".join(r["source_flags"]),
} for r in results]).to_csv(OUT / "review.csv", index=False, encoding="utf-8-sig")

print("\n========== TỔNG KẾT ==========")
for branch in dict.fromkeys(r["branch"] for r in results):
    items = [r for r in results if r["branch"] == branch]
    print(
        f"/{branch}/ | mẫu={len(items)}"
        f" | lỗi={sum('PARSE_ERROR' in r['flags'] for r in items)}"
        f" | bố cục tự đoán={sum('GUESSED_LAYOUT_REVIEW' in r['flags'] for r in items)}"
        f" | bỏ tóm tắt={sum(r['summary_removed'] for r in items)}"
        f" | còn nhiễu={sum('REMAINING_NOISE' in r['flags'] for r in items)}"
        f" | encoding cần review={sum(any('ENCODING' in f for f in r['flags']) for r in items)}"
        f" | có ngày={sum(bool(r['date_candidates']) for r in items)}"
        f" | có cờ nguồn={sum(bool(r['source_flags']) for r in items)}"
    )

print("\n========== MẪU MỖI NHÁNH ==========")
for branch in dict.fromkeys(r["branch"] for r in results):
    row = next((r for r in results if r["branch"] == branch and r["text"]), None)
    if row:
        print("\n", row["url"])
        print("TITLE:", row.get("title"))
        print("ENCODING:", row.get("encoding"), "| SELECTOR:", row["selector"])
        print("FLAGS:", row["flags"], "| SOURCE:", row["source_flags"])
        print("ĐẦU:\n", row["text"][:650])
        print("CUỐI:\n", row["text"][-250:])

print("\n========== KIỂM TRA RIÊNG BÀI TỪNG LỖI ENCODING ==========")
for row in results:
    if "/fuke/nxjyss/12564.html" in row["url"]:
        print("Tiêu đề cũ:", row.get("old_title"))
        print("Tiêu đề mới:", row.get("title"))
        print("Encoding ứng viên:", row.get("encoding_candidates"))
        print("Nội dung:", row["text"][:500])

print("\n========== BỐ CỤC TỰ ĐOÁN: CẦN REVIEW ==========")
for row in results:
    if "GUESSED_LAYOUT_REVIEW" in row["flags"]:
        print(row["url"], "|", row["selector"])
        print(json.dumps(row.get("body_candidates", []), ensure_ascii=False))
        print("ĐẦU:", row["text"][:200])

print("\nĐã lưu:", OUT)
print("Không có cờ không đồng nghĩa đã xác nhận nội dung đầy đủ hoặc đúng y khoa.")

Note: you may need to restart the kernel to use updated packages.
[1/120] /dise/ | utf-8 | div.detailc | 3126 ký tự | [] | nguồn=[]
[2/120] /dise/ | utf-8 | div.detailc | 1120 ký tự | [] | nguồn=[]
[3/120] /dise/ | utf-8 | div.detailc | 1113 ký tự | [] | nguồn=[]
[4/120] /dise/ | utf-8 | div.detailc | 693 ký tự | [] | nguồn=[]
[5/120] /dise/ | utf-8 | div.detailc | 1115 ký tự | ['DUPLICATE_PARAGRAPH_REVIEW'] | nguồn=[]
[6/120] /dise/ | utf-8 | div.detailc | 1055 ký tự | [] | nguồn=[]
[7/120] /dise/ | utf-8 | div.detailc | 708 ký tự | [] | nguồn=[]
[8/120] /dise/ | utf-8 | div.detailc | 792 ký tự | [] | nguồn=[]
[9/120] /dise/ | utf-8 | div.detailc | 399 ký tự | [] | nguồn=[]
[10/120] /dise/ | utf-8 | div.detailc | 734 ký tự | [] | nguồn=[]
[11/120] /dise/ | utf-8 | div.detailc | 514 ký tự | [] | nguồn=[]
[12/120] /dise/ | utf-8 | div.detailc | 710 ký tự | [] | nguồn=[]
[13/120] /dise/ | utf-8 | div.detailc | 687 ký tự | [] | nguồn=[]
[14/120] /dise/ | utf-8 | div.detailc | 12987 ký tự 

In [16]:
%pip -q install beautifulsoup4

import re, json, hashlib
from pathlib import Path
from collections import Counter
import pandas as pd
from bs4 import BeautifulSoup

ROOT = Path("/kaggle/working")
HTML_DIR = ROOT / "cnkang_review_v4"
PREVIOUS = ROOT / "cnkang_review_v5"
OUT = ROOT / "cnkang_review_v6"
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(path):
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

jobs = read_jsonl(HTML_DIR / "review.jsonl")
previous = {r["key"]: r for r in read_jsonl(PREVIOUS / "review.jsonl")}
if not jobs:
    raise FileNotFoundError("Không tìm thấy HTML/manifest vòng v4.")

def compact(text):
    return re.sub(r"\s+", "", text)

def cjk_ratio(text):
    return len(re.findall(r"[\u4e00-\u9fff]", text)) / max(len(compact(text)), 1)

def render(node):
    clone = BeautifulSoup(str(node), "html.parser")
    for tag in clone.select("script,style,nav,footer,aside,form"):
        tag.decompose()
    for tag in clone.find_all(["p", "div", "li", "h1", "h2", "h3", "tr", "br"]):
        tag.insert_before("\n")
        tag.insert_after("\n")
    return "\n".join(
        line for line in (
            re.sub(r"[ \t\u3000]+", " ", s).strip()
            for s in clone.get_text("", strip=False).splitlines()
        ) if line
    )

def node_label(node):
    return (
        node.name
        + ("#" + node["id"] if node.get("id") else "")
        + ("." + ".".join(node.get("class", [])) if node.get("class") else "")
    )

def decode_html(raw):
    head = raw[:16000].decode("latin1")
    match = re.search(r'charset\s*=\s*["\']?\s*([\w-]+)', head, re.I)
    declared = match.group(1).lower() if match else ""
    candidates = []
    for enc in dict.fromkeys([declared, "utf-8", "gb18030", "gbk", "big5"]):
        if not enc:
            continue
        try:
            html = raw.decode(enc, errors="strict")
        except (UnicodeError, LookupError):
            continue
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup.select("script,style"):
            tag.decompose()
        text = soup.get_text(" ", strip=True)
        ratio = cjk_ratio(text)
        mojibake = len(re.findall(r"[ÃÂÕÈÊÐÖØËÄÎÏÒÓº¼½¾]", text))
        pua = len(re.findall(r"[\ue000-\uf8ff]", text))
        score = (
            ratio * 100
            - mojibake / max(len(text), 1) * 150
            - pua / max(len(text), 1) * 200
            + (5 if enc == declared else 0)
            + (10 if enc == "utf-8" else 0)
        )
        candidates.append((score, enc, html))
    if not candidates:
        return raw.decode("utf-8", errors="replace"), "utf-8-replace", []
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][2], candidates[0][1], [
        {"encoding": enc, "score": round(score, 2)}
        for score, enc, _ in candidates
    ]

DATE_RE = re.compile(r"(?:19|20)\d{2}[-/.年]\d{1,2}[-/.月]\d{1,2}日?")
NOISE = [
    "核心提示", "您的位置", "手机查看", "扫描二维码", "凑数的文字",
    "相关文章", "免费提问", "立即咨询", "上一篇", "下一篇",
    "精彩推荐", "最新文章排行", "月排行", "总排行",
    "医学药学", "专题推荐", "推荐专题",
]
UI_LINES = {
    "相关文章", "免费提问", "立即咨询", "精彩推荐", "最新文章排行",
    "搞笑", "女性", "生活", "热点", "手机查看",
}

def choose_body(soup):
    matches = soup.select(".detailc")
    matches += soup.find_all(id=re.compile(r"^endtext$", re.I))
    known = []
    for node in matches:
        if not any(node is p for p in known):
            known.append(node)
    if len(known) == 1:
        body = known[0]
        scope = body.find_parent(class_="detail") or body.find_parent(id="wleft")
        return body, scope, node_label(body), []

    conmain = soup.select(".conmain")
    if len(conmain) != 1:
        return None, None, "", []

    scope = conmain[0]
    whole_n = len(compact(scope.get_text()))
    candidates = []
    # Tìm div con đủ bao phủ nội dung, tránh lấy cả tiêu đề/ngày của .conmain.
    for node in scope.find_all(["div", "section", "article"]):
        n = len(compact(node.get_text()))
        if n < 40:
            continue
        link_n = sum(len(compact(a.get_text())) for a in node.find_all("a"))
        coverage = n / max(whole_n, 1)
        link_ratio = link_n / max(n, 1)
        if coverage >= 0.72 and link_ratio < 0.3:
            candidates.append((n, node, coverage, link_ratio))
    candidates.sort(key=lambda x: x[0])

    info = [{
        "node": node_label(node), "chars": n,
        "coverage": round(coverage, 3), "link_ratio": round(ratio, 3)
    } for n, node, coverage, ratio in candidates[:5]]

    if candidates:
        body = candidates[0][1]
        return body, scope, ".conmain > " + node_label(body), info
    # Giữ vùng rộng để review, không coi là đã tách thân bài thành công.
    return scope, scope, ".conmain_UNSPLIT", info

def remove_summary(text, title):
    # Chỉ cắt khi có bằng chứng phần mở đầu của tóm tắt được lặp lại.
    mapping = list(re.finditer(r"\S", text))
    normalized = "".join(m.group() for m in mapping)
    marker = re.search(r"核心提示[:：]?", normalized[:100])
    actions = []
    if marker:
        start = marker.end()
        title_n = compact(title)
        if title_n and normalized.startswith(title_n, start):
            start += len(title_n)
        cutoff = re.search(r"…+|\.{3,}", normalized[start:start + 700])
        if cutoff:
            end_summary = start + cutoff.start()
            after_cut = start + cutoff.end()
            summary = normalized[start:end_summary]
            anchor = summary[:30]
            if len(anchor) >= 20:
                repeat = normalized.find(
                    anchor, after_cut, min(len(normalized), after_cut + 400)
                )
                if repeat >= after_cut:
                    pos = mapping[repeat].start()
                    actions.append({
                        "action": "remove_repeated_summary",
                        "removed": text[:pos], "anchor": anchor
                    })
                    text = text[pos:].lstrip()
    kept = []
    for line in text.splitlines():
        if line.strip() in UI_LINES:
            actions.append({"action": "remove_ui_line", "removed": line})
        else:
            kept.append(line)
    return "\n".join(kept).strip(), actions

def corruption_review(text):
    evidence = []
    for index, paragraph in enumerate(text.splitlines(), 1):
        if not paragraph.strip():
            continue
        reasons = []
        pua = len(re.findall(r"[\ue000-\uf8ff\U000F0000-\U0010FFFF]", paragraph))
        if pua:
            reasons.append(f"private_unicode={pua}")
        if "\ufffd" in paragraph:
            reasons.append("replacement_character")
        if re.search(r"(?i)(?:[?？]/?[a-z]{1,6}>|</?(?:p|div|span|br)>)", paragraph):
            reasons.append("broken_html_fragment")
        latin_bad = len(re.findall(r"[ÃÂÕÈÊÐÖØËÄÎÏÒÓº¼½¾]", paragraph))
        if latin_bad >= 4:
            reasons.append("possible_latin_mojibake")
        if (
            len(compact(paragraph)) >= 40
            and cjk_ratio(paragraph) < 0.15
            and re.search(r"[^\x00-\x7f]", paragraph)
        ):
            reasons.append("low_cjk_paragraph")
        if reasons:
            match = re.search(r"[\ue000-\uf8ff\ufffd]", paragraph)
            pos = match.start() if match else 0
            evidence.append({
                "paragraph": index, "reasons": reasons,
                "excerpt": paragraph[max(0, pos-60):pos+180]
            })
    return evidence

def order_review(text):
    cn = dict(zip("一二三四五六七八九十", range(1, 11)))
    patterns = {
        "cn_brackets": r"[（(]([一二三四五六七八九十])[）)]",
        "num_brackets": r"[（(](\d{1,2})[）)]",
        "cn_heading": r"^\s*([一二三四五六七八九十])[、．.]",
        "num_heading": r"^\s*(\d{1,2})(?:[、．.]|\s+)",
    }
    sequences = {}
    for name, pattern in patterns.items():
        values = []
        for line in text.splitlines():
            match = re.search(pattern, line[:120])
            if match:
                token = match.group(1)
                values.append(int(token) if token.isdigit() else cn[token])
        sequences[name] = values
    descending = any(
        values[i] > values[i+1] > values[i+2]
        for values in sequences.values() for i in range(len(values)-2)
    )
    return descending, sequences

results = []
for i, old in enumerate(jobs, 1):
    key = old["key"]
    row = {
        "url": old["url"], "branch": old["branch"], "key": key,
        "flags": [], "source_flags": [], "text": "", "chars": 0,
        "date_candidates": [], "selector": "",
    }
    try:
        raw = (HTML_DIR / f"{key}.html").read_bytes()
        html, encoding, scores = decode_html(raw)
        row.update(encoding=encoding, encoding_candidates=scores)
        soup = BeautifulSoup(html, "html.parser")
        body, scope, selector, candidates = choose_body(soup)
        row.update(selector=selector, body_candidates=candidates)
        if body is None:
            raise ValueError("Chưa xác định được vùng thân bài")
        if selector == ".conmain_UNSPLIT":
            row["flags"].append("BODY_SCOPE_REVIEW")
        if selector.startswith(".conmain"):
            row["flags"].append("CONMAIN_VALIDATION_SAMPLE")

        heading = (
            scope.find("h1") if scope is not None else None
        ) or soup.find("h1")
        html_title = soup.title.get_text(" ", strip=True) if soup.title else ""
        row["title"] = (
            heading.get_text(" ", strip=True) if heading
            else html_title.split("_")[0].strip()
        )

        before = render(body)
        text, actions = remove_summary(before, row["title"])
        row.update(
            text=text, chars=len(text), cleaning_actions=actions,
            summary_removed=any(a["action"] == "remove_repeated_summary" for a in actions)
        )

        if scope is not None and scope is not body:
            scope_text = compact(scope.get_text(" ", strip=True))
            body_text = compact(body.get_text(" ", strip=True))
            position = scope_text.find(body_text)
            if position >= 0:
                prefix = scope_text[:position][-600:]
                row["date_candidates"] = list(dict.fromkeys(DATE_RE.findall(prefix)))
                row["date_context"] = prefix[-300:]
        if not row["date_candidates"]:
            row["flags"].append("DATE_MISSING")

        noise = [x for x in NOISE if x in text]
        row["noise_markers"] = noise
        if noise:
            row["flags"].append("REMAINING_NOISE")
        if not text:
            row["flags"].append("EMPTY_TEXT")
        elif len(compact(text)) < 200:
            row["flags"].append("SHORT_TEXT_REVIEW")

        corrupt = corruption_review(text)
        row["corruption_evidence"] = corrupt
        if corrupt:
            row["flags"].append("LOCAL_CORRUPTION_REVIEW")

        descending, sequences = order_review(text)
        row["number_sequences"] = sequences
        if descending:
            row["flags"].append("DESCENDING_ORDER_REVIEW")

        paragraphs = [compact(p) for p in text.splitlines() if len(compact(p)) >= 40]
        if any(n > 1 for n in Counter(paragraphs).values()):
            row["flags"].append("DUPLICATE_PARAGRAPH_REVIEW")

        for topic in ["乳腺癌", "卵巢癌", "糖尿病", "淋巴瘤", "子宫内膜异位症"]:
            if topic in row["title"] and topic not in compact(text):
                row["flags"].append("TITLE_TOPIC_REVIEW")
                break

        # So sánh v4 và v5 bằng độ dài đã bỏ khoảng trắng.
        refs = {"v4": old.get("text", ""), "v5": previous.get(key, {}).get("text", "")}
        changes = {}
        for version, ref in refs.items():
            old_n, new_n = len(compact(ref)), len(compact(text))
            ratio = new_n / max(old_n, 1)
            changes[version] = {
                "old_chars": old_n, "new_chars": new_n, "ratio": round(ratio, 3)
            }
            if old_n >= 200 and ratio < 0.6:
                row["flags"].append(f"LARGE_DROP_VS_{version.upper()}")
        row["length_comparison"] = changes

        patterns = {
            "PROMOTIONAL_CLAIM_REVIEW":
                r"无毒副作用|无任何副作用|纯天然绿色饮品|包治|保证治愈",
            "PRODUCT_PROMOTION_REVIEW":
                r"长期饮用.{0,25}茶|万能膏|购买|订购|咨询热线",
            "PERSONAL_EXPERIENCE_REVIEW":
                r"笔者|帅哥一枚|我就发现|我的经验|我自己",
        }
        row["source_evidence"] = {}
        for flag, pattern in patterns.items():
            matches = list(re.finditer(pattern, text))
            if matches:
                row["source_flags"].append(flag)
                row["source_evidence"][flag] = [
                    text[max(0, m.start()-35):m.end()+60] for m in matches[:3]
                ]

        if text.rstrip().endswith(("……", "...")):
            row["flags"].append("POSSIBLE_TRUNCATION_REVIEW")
        row["hash"] = hashlib.sha256(compact(text).encode()).hexdigest() if text else None
        (OUT / f"{key}_before.txt").write_text(before, encoding="utf-8")
        (OUT / f"{key}_clean.txt").write_text(text, encoding="utf-8")

    except Exception as e:
        row["flags"].append("PARSE_ERROR")
        row["error"] = str(e)

    results.append(row)
    print(f"[{i}/{len(jobs)}] /{row['branch']}/ | {row['selector']} | "
          f"{row['chars']} ký tự | {row['flags']}")

counts = Counter(r.get("hash") for r in results if r.get("hash"))
for row in results:
    if row.get("hash") and counts[row["hash"]] > 1:
        row["flags"].append("DUPLICATE_DOCUMENT_REVIEW")

(OUT / "review.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in results), encoding="utf-8"
)
pd.DataFrame([{
    "url": r["url"], "branch": r["branch"],
    "selector": r["selector"], "title": r.get("title", ""),
    "encoding": r.get("encoding", ""), "chars": r["chars"],
    "dates": "; ".join(r["date_candidates"]),
    "flags": "; ".join(r["flags"]),
    "source_flags": "; ".join(r["source_flags"]),
} for r in results]).to_csv(OUT / "review.csv", index=False, encoding="utf-8-sig")

print("\n========== TỔNG KẾT ==========")
for branch in dict.fromkeys(r["branch"] for r in results):
    items = [r for r in results if r["branch"] == branch]
    print(
        f"/{branch}/ | mẫu={len(items)}"
        f" | lỗi={sum('PARSE_ERROR' in r['flags'] for r in items)}"
        f" | còn nhiễu={sum('REMAINING_NOISE' in r['flags'] for r in items)}"
        f" | lỗi chữ nghi ngờ={sum('LOCAL_CORRUPTION_REVIEW' in r['flags'] for r in items)}"
        f" | giảm mạnh={sum(any(f.startswith('LARGE_DROP') for f in r['flags']) for r in items)}"
        f" | có ngày={sum(bool(r['date_candidates']) for r in items)}"
        f" | có cờ nguồn={sum(bool(r['source_flags']) for r in items)}"
    )

print("\n========== CONMAIN: TIÊU ĐỀ / NGÀY / THÂN BÀI ==========")
for row in results:
    if row["selector"].startswith(".conmain"):
        print("\n", row["url"])
        print("TITLE:", row.get("title"), "| DATE:", row["date_candidates"])
        print("SELECTOR:", row["selector"], "| FLAGS:", row["flags"])
        print("ĐẦU:", row["text"][:350])
        print("CUỐI:", row["text"][-180:])

print("\n========== ĐOẠN NGHI LỖI CHỮ ==========")
for row in results:
    if row.get("corruption_evidence"):
        print("\n", row["url"], "| ENCODING:", row.get("encoding"))
        print(json.dumps(row["corruption_evidence"][:5], ensure_ascii=False, indent=2))

print("\n========== CÁC TRANG GIẢM NỘI DUNG MẠNH ==========")
for row in results:
    if any(f.startswith("LARGE_DROP") for f in row["flags"]):
        print("\n", row["url"])
        print("ĐỘ DÀI:", row.get("length_comparison"))
        print("SỐ KHỐI ĐÃ LOẠI:", len(row.get("cleaning_actions", [])))
        print("ĐẦU MỚI:", row["text"][:200])
        print("CUỐI MỚI:", row["text"][-200:])

print("\nĐã lưu:", OUT)
print("CONMAIN_VALIDATION_SAMPLE là cờ kiểm chứng bố cục, chưa phải lỗi.")
print("Không tự đảo đoạn, sửa chữ hỏng hoặc loại bài chỉ vì ngắn.")

Note: you may need to restart the kernel to use updated packages.
[1/120] /dise/ | div.detailc | 3126 ký tự | ['LOCAL_CORRUPTION_REVIEW']
[2/120] /dise/ | div.detailc | 1120 ký tự | []
[3/120] /dise/ | div.detailc | 1113 ký tự | []
[4/120] /dise/ | div.detailc | 693 ký tự | []
[5/120] /dise/ | div.detailc | 1115 ký tự | ['DUPLICATE_PARAGRAPH_REVIEW']
[6/120] /dise/ | div.detailc | 1055 ký tự | []
[7/120] /dise/ | div.detailc | 708 ký tự | []
[8/120] /dise/ | div.detailc | 792 ký tự | []
[9/120] /dise/ | div.detailc | 399 ký tự | []
[10/120] /dise/ | div.detailc | 734 ký tự | []
[11/120] /dise/ | div.detailc | 514 ký tự | []
[12/120] /dise/ | div.detailc | 710 ký tự | []
[13/120] /dise/ | div.detailc | 687 ký tự | []
[14/120] /dise/ | div.detailc | 12987 ký tự | ['LOCAL_CORRUPTION_REVIEW', 'DESCENDING_ORDER_REVIEW']
[15/120] /dise/ | div.detailc | 695 ký tự | []
[16/120] /dise/ | div.detailc | 305 ký tự | []
[17/120] /dise/ | div.detailc | 976 ký tự | []
[18/120] /dise/ | div.detailc | 